<a href="https://colab.research.google.com/github/Aaryan8597/AI-ML-AaryanLamichhane2408597-/blob/main/2025_W08_Text_Classification_Question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Necessary imports

In [41]:
import numpy as np
import pandas as pd
import re
import nltk
nltk.download('punkt_tab')
from nltk import word_tokenize
from nltk.tokenize import RegexpTokenizer
from nltk.tokenize import RegexpTokenizer
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize,pos_tag
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


# Loading the Dataset

In [4]:
df = pd.read_csv('/content/drive/MyDrive/AIML/Worksheet8/trum_tweet_sentiment_analysis.csv')

In [7]:
# making sure that the dataset contains at least two columns
df = df[['text', 'Sentiment']]

In [8]:
# removing the duplicate values
df.dropna()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0
...,...,...
1850118,Everytime im like 'How the fuck I follow Melan...,0
1850119,RT @imgur: The Trump Handshake. https://t.co/R...,0
1850120,"""Greenspan warns Trump's policies risk inflati...",0
1850121,RT @FasinatingLogic: We must also #INVESTIGATE...,1


In [9]:
df.head()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


# Text cleaning and Tokenization

## lowercasing the text

In [13]:
def make_lowercase(input_text):
  clean_text = input_text.lower()
  return clean_text

## Removing URLs, mentions, punctuation, and special characters

In [15]:
def remove_links(input_text):
  url_pattern = r'http\S+|www\S+'
  clean_text = re.sub(url_pattern, '', input_text)
  return clean_text

In [17]:
#removing emoji
def remove_emoji(text_with_emoji):
  emoji_pattern = re.compile("["
                            u"\U0001F600-\U0001F64F"
                            u"\U0001F300-\U0001F5FF"
                            u"\U0001F680-\U0001F6FF"
                            u"\U0001F1E0-\U0001F1FF"
                            u"\U00002702-\U000027B0"
                            u"\U000024C2-\U0001F251"
                            "]+", flags=re.UNICODE)

  clean_text = emoji_pattern.sub(r' ', text_with_emoji)
  return clean_text

In [20]:
# removing unwanted character
def remove_unwanted_characters(text):
  text = re.sub("@[A-Za-z0-9_]+", " ", text)
  text = re.sub("#[A-Za-z0-9_]+", "", text)
  text = re.sub("[^0-9A-Za-z ]", "", text)
  text = remove_emoji(text)
  text = text.replace('  ', ' ')
  return text.strip()

#testing
test = "RT @TwitterUser: I love the USA! 🇺🇸 #America #News http://test.com"
lowercase = make_lowercase(test)
remove_link = remove_links(lowercase)
unwanted = remove_unwanted_characters(remove_link)
print(f"Original: {test}")
print(f"Cleaned:  {unwanted}")

Original: RT @TwitterUser: I love the USA! 🇺🇸 #America #News http://test.com
Cleaned:  rt  i love the usa


In [26]:
# removing punctuation
def remove_punctuation(text):
  clean_text = re.sub("[^a-zA-Z ]", "", text)
  return clean_text

## Removing Stopwords

In [24]:
stop_words = set(stopwords.words('english'))
stop_words.update(['RT', 'rt', '@'])
def remove_stopwords(text):
  stop_word = []
  for word in text:
    if word not in stop_words:
      stop_word.append(word)
  return stop_word

#testing
test_inputs = ['RT', '@', 'The', 'election', 'is', 'a', 'very', 'important', 'moment', 'for', 'the', 'entire', 'country', '.']
print("Original Tokens:")
print(test_inputs)
tokens_without_stopwords = remove_stopwords(test_inputs)
print("\nTokens After Removing Stopwords:")
print(tokens_without_stopwords)


Original Tokens:
['RT', '@', 'The', 'election', 'is', 'a', 'very', 'important', 'moment', 'for', 'the', 'entire', 'country', '.']

Tokens After Removing Stopwords:
['The', 'election', 'important', 'moment', 'entire', 'country', '.']


## Tokenization

### Lemmatization

In [29]:
def apply_lemmatization(word_list):
  lemmatizer = WordNetLemmatizer()
  clean_root_words = []
  for word in word_list:
    root = lemmatizer.lemmatize(word, pos='v')
    clean_root_words.append(root)
  return clean_root_words

### Stemming

In [33]:
def apply_stemming(word_list):
  stemmer = PorterStemmer()
  stemmed_words = []
  for word in word_list:
    root = stemmer.stem(word)
    stemmed_words.append(root)
  return stemmed_words

# Build a Text Cleaning Pipeline

In [35]:
def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This...
  """
  # Convert the input to small/lower order.
  data = make_lowercase(dataset)
  # Remove URLs
  data = remove_links(data)
  # Remove emojis
  data = remove_emoji(data)
  # Remove all other unwanted characters.
  data = remove_unwanted_characters(data)
  # Remove punctuation.
  data = remove_punctuation(data)
  # Create tokens.
  tokens = data.split()
  # Remove stopwords:
  tokens = remove_stopwords(tokens)
  if rule == "lemmatize":
    tokens = apply_lemmatization(tokens)
  elif rule == "stem":
    tokens = apply_stemming(tokens)
  else:
    print("Pick between lemmatize or stem")

  return " ".join(tokens)


# Cleaning the data

In [36]:
df['cleaned_text'] = df['text'].apply(lambda x: text_cleaning_pipeline(x, rule="lemmatize"))
df = df.dropna(subset=['cleaned_text'])

# Train-Test Split

In [39]:
X = df['cleaned_text']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#checking the rows
print(f"Original rows: {len(df)}")
print(f"Training rows: {len(X_train)}")
print(f"Testing rows:  {len(X_test)}")

Original rows: 1850123
Training rows: 1480098
Testing rows:  370025


# TF-IDF Vectorization

In [40]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=500000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print(f"X_train_tfidf shape: {X_train_tfidf.shape}")
print(f"X_test_tfidf shape:  {X_test_tfidf.shape}")

X_train_tfidf shape: (1480098, 500000)
X_test_tfidf shape:  (370025, 500000)


# Training Model

In [44]:
model_logistic = LogisticRegression()
model_logistic.fit(X_train_tfidf, y_train)
y_pred = model_logistic.predict(X_test_tfidf)
print("\n EVALUATION REPORT")
print(classification_report(y_test, y_pred))


 EVALUATION REPORT
              precision    recall  f1-score   support

           0       0.95      0.97      0.96    248563
           1       0.94      0.90      0.92    121462

    accuracy                           0.95    370025
   macro avg       0.95      0.94      0.94    370025
weighted avg       0.95      0.95      0.95    370025



#

#